In [ ]:
!pip install mistralai

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload() 
    pdf_path = list(uploaded.keys())[0]
    print("Using PDF:", pdf_path)

except ImportError:
    pdf_path = "data/benchmark/flexqube_arsredovisning_2022.pdf" 
    
    print(f"Running locally, using: {pdf_path}")

In [ ]:
MISTRAL_API_KEY = "mistralocr_apikey"

In [ ]:
from mistralai import Mistral
from pathlib import Path

client = Mistral(api_key=MISTRAL_API_KEY)

# Upload PDF to Mistral
with open(pdf_path, "rb") as f:
    upload = client.files.upload(
        file={"file_name": Path(pdf_path).name, "content": f.read()},
        purpose="ocr"
    )

file_id = upload.id
print("Uploaded file id:", file_id)

In [ ]:
from mistralai import DocumentURLChunk

# Get signed URL for uploaded PDF
signed_url = client.files.get_signed_url(file_id=file_id, expiry=60).url

# Run OCR
ocr_response = client.ocr.process(
    model="mistral-ocr-latest",
    document=DocumentURLChunk(document_url=signed_url),
    include_image_base64=True
)

# Extract Markdown text
all_md = []
for i, page in enumerate(ocr_response.pages):
    all_md.append(f"# Page {i + 1}\n\n{page.markdown}")

md_text = "\n\n".join(all_md)

# Save to file
with open("mistral_output.md", "w", encoding="utf-8") as f:
    f.write(md_text)

print("OCR finished — saved to mistral_output.md")

In [ ]:
from google.colab import files

# Download the OCR result
files.download("mistral_output.md")